In [0]:
%sql
create or replace view workspace.pj_sales.vw_rank_by_seller as (
  with
    aggregated_seller as (
      select
         tfs._sk_seller
        ,tds.seller_id
        ,tds.seller_name
        ,count(distinct tfs.invoice_id)                   as sales_quantity
        ,round(sum(tfs.part_total_price), 2)              as billing
        ,round(sum(tfs.part_profit), 2)                   as profit
        ,round(sum(tfs.part_quantity), 2)                 as product_quantity
      from workspace.pj_sales.tb_fact_sale_gold tfs
      inner join workspace.pj_sales.tb_dim_seller_gold tds
        on tds._sk_seller = tfs._sk_seller
      group by
         tfs._sk_seller
        ,tds.seller_id
        ,tds.seller_name
    )
  select
     ags.seller_id
    ,ags.seller_name
    ,ags.sales_quantity
    ,ags.billing
    ,ags.profit
    ,ags.product_quantity
    ,dense_rank() over (order by ags.billing        desc)   as rank_by_billing
    ,dense_rank() over (order by ags.profit         desc)   as rank_by_profit
    ,dense_rank() over (order by ags.sales_quantity desc)   as rank_by_sales_quantity
  from aggregated_seller ags
)